# Jupyter Notebook — 2015–2025 verilerini birleştirme

python -m sgparl --from 2026-01-01 --to 2026-12-31 --workers 5 --output data/parliament_2026 --format csv


In [6]:
import pandas as pd
from pathlib import Path

BASE = Path("/Users/kemalgunay/PycharmProjects/sgparl/data")
all_years = []
summary = []

for year in range(2015, 2026):

    folder = BASE / f"parliament_{year}"

    speeches = pd.read_csv(folder / "speeches.csv")
    topics = pd.read_csv(folder / "topics.csv")

    # Konuşmaları gündem maddeleriyle eşleştir
    merged = speeches.merge(
        topics[["topic_id", "title", "section_type"]],
        on="topic_id",
        how="left",
        validate="many_to_one",
        indicator=True
    )

    # Yıl bilgisi
    merged["year"] = year

    # Eşleşme kontrolü
    unmatched = (merged["_merge"] == "left_only").sum()

    summary.append({
        "year": year,
        "speeches": len(speeches),
        "topics": len(topics),
        "merged": len(merged),
        "unmatched": unmatched
    })

    merged = merged.drop(columns="_merge")
    all_years.append(merged)

# Bütün yılları birleştir
df = pd.concat(all_years, ignore_index=True)

# Sonuçları göster
summary_df = pd.DataFrame(summary)

display(summary_df)

print("Toplam konuşma:", len(df))
print("Yıl sayısı:", df["year"].nunique())
print("Tekil speech_id:", df["speech_id"].nunique())

,year,speeches,topics,merged,unmatched
0,2015,3490,665,3490,0
1,2016,5894,1180,5894,0
2,2017,6222,1262,6222,0
3,2018,6572,1231,6572,0
4,2019,6902,1248,6902,0
5,2020,6538,1319,6538,0
6,2021,10049,2434,10049,0
7,2022,10650,2611,10650,0
8,2023,11213,2301,11213,0
9,2024,9973,2297,9973,0


Toplam konuşma: 86736
Yıl sayısı: 11
Tekil speech_id: 86736


In [7]:
df

,date,speech_id,topic_id,speech_order,member_name_original,member_name,text,num_words,num_characters,num_sentences,num_syllables,party,gender,is_appointment,is_chairing,is_noise,title,section_type,year
0,2015-01-19,2015-01-19-T-002-S-00001,2015-01-19-T-002,1,NaN,NaN,[() Order for Second Reading read. ()],7,25,2,15,NaN,NaN,False,False,True,Pawnbrokers Bill,BI,2015
1,2015-01-19,2015-01-19-T-002-S-00002,2015-01-19-T-002,2,The Senior Minister of State for Law (Ms Indra...,Indranee Rajah,"Mdm Speaker, I beg to move, ""That the Bill be ...",1778,8607,72,2992,PAP,F,True,False,False,Pawnbrokers Bill,BI,2015
2,2015-01-19,2015-01-19-T-002-S-00003,2015-01-19-T-002,3,Mdm Speaker,Speaker,Assoc Prof Fatimah Lateef.,4,22,1,9,NaN,NaN,True,False,False,Pawnbrokers Bill,BI,2015
3,2015-01-19,2015-01-19-T-002-S-00004,2015-01-19-T-002,4,Assoc Prof Fatimah Lateef (Marine Parade),Fatimah Lateef,"Mdm Speaker, I stand in support of the Bill. I...",532,2586,24,924,PAP,F,False,False,False,Pawnbrokers Bill,BI,2015
4,2015-01-19,2015-01-19-T-002-S-00005,2015-01-19-T-002,5,Mr Yee Jenn Jong (Non-Constituency Member),Yee Jenn Jong,This Bill seeks to update the Pawnbrokers Act ...,517,2363,24,803,WP,M,False,False,False,Pawnbrokers Bill,BI,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86731,2025-11-06,2025-11-06-T-098-S-00367,2025-11-06-T-098,367,Mr Cai Yinzhou,Cai Yinzhou,asked the Minister for Social and Family Devel...,59,356,1,125,PAP,M,False,False,False,Safeguards and Support for Frontline Professio...,WANA,2025
86732,2025-11-06,2025-11-06-T-098-S-00368,2025-11-06-T-098,368,Mr Low Wu Yang Andre,Low Wu Yang Andre,asked the Minister for Social and Family Devel...,68,370,1,133,WP,M,False,False,False,Safeguards and Support for Frontline Professio...,WANA,2025
86733,2025-11-06,2025-11-06-T-098-S-00369,2025-11-06-T-098,369,Mr Masagos Zulkifli B M M,Masagos Zulkifli B M M,This question has been answered as part of the...,27,137,2,50,PAP,M,False,False,False,Safeguards and Support for Frontline Professio...,WANA,2025
86734,2025-11-06,2025-11-06-T-099-S-00370,2025-11-06-T-099,370,NaN,NaN,[() The following statement was made in a repl...,39,161,1,67,NaN,NaN,False,False,True,Clarification by Senior Minister of State for ...,OS,2025


In [8]:
df.shape

(86736, 19)

Bu kod her yılın speeches.csv dosyasını aynı yılın topics.csv dosyasıyla topic_id üzerinden eşleştirir. Ayrıca gözlem kaybı veya eşleşmeyen kayıt olup olmadığını kontrol eder.
# Veriyi kaydetme

In [11]:
OUTPUT = Path("data/combined")
OUTPUT.mkdir(parents=True, exist_ok=True)

df.to_csv(
    OUTPUT / "singapore_parliament_2015_2025.csv",
    index=False
)

# Cinsiyet analizi için ayrı veri seti

In [9]:
df_clean = df[
    (df["is_noise"] == False) &
    (df["gender"].isin(["M", "F"])) &
    (df["member_name"].notna())
].copy()

print("Orijinal konuşmalar:", len(df))
print("Analize uygun konuşmalar:", len(df_clean))

print("\nCinsiyet dağılımı:")
display(df_clean["gender"].value_counts())

Orijinal konuşmalar: 86736
Analize uygun konuşmalar: 65429

Cinsiyet dağılımı:


gender
M    50821
F    14608
Name: count, dtype: int64

# Cinsiyet analizi için ayrı veri seti

In [13]:
df_gender = df[
    (df["is_noise"] == False) &
    (df["gender"].isin(["M", "F"])) &
    (df["member_name"].notna())
].copy()

df_gender.to_csv(
    OUTPUT / "singapore_gender_2015_2025.csv",
    index=False
)

print("Toplam analiz gözlemi:", len(df_gender))

Toplam analiz gözlemi: 65429


# İlk betimsel analiz
Yıllara göre kadın ve erkek konuşmacıların konuşma sayılarını karşılaştıralım.

In [14]:
gender_year = pd.crosstab(
    df_gender["year"],
    df_gender["gender"]
)

gender_year["total"] = gender_year.sum(axis=1)

gender_year["female_share"] = (
    gender_year["F"] / gender_year["total"] * 100
).round(2)

display(gender_year)

gender,F,M,total,female_share
year,,,,
2015,585,2168,2753,21.25
2016,791,3612,4403,17.97
2017,819,3795,4614,17.75
2018,1068,3623,4691,22.77
2019,1187,3696,4883,24.31
2020,1237,3551,4788,25.84
2021,1752,6148,7900,22.18
2022,1690,6594,8284,20.40
2023,1857,6492,8349,22.24


Bu bize kadın konuşmacılara ait konuşmaların toplam konuşmalar içindeki payının 2015–2025 arasında nasıl değiştiğini gösterecek. Ancak bu oran, kadın milletvekillerinin parlamentodaki temsil oranıyla aynı şey değildir.
Önemli: Birleştirme sonrasında summary_df tablosundaki unmatched değerlerinin sıfır olmasını ve her yıl için speeches ile merged sayılarının eşit olmasını bekliyoruz.
Önce bu kodu çalıştırıp summary_df çıktısını paylaşmanı öneriyorum. Ardından 11 yıllık verinin kalitesini değerlendirebilir ve araştırma tasarımına geçebiliriz.

In [15]:
topic_summary = (
    df_clean.groupby(["title", "gender"])
    .size()
    .unstack(fill_value=0)
)

topic_summary["total"] = topic_summary.sum(axis=1)

display(
    topic_summary.sort_values(
        "total", ascending=False
    ).head(20)
)

gender,F,M,total
title,,,
Debate on Annual Budget Statement,261,819,1080
"Committee of Supply – Head X (Ministry of Culture, Community and Youth)",178,544,722
Committee of Supply – Head S (Ministry of Manpower),165,515,680
Committee of Supply – Head T (Ministry of National Development),150,488,638
Committee of Supply – Head K (Ministry of Education),184,408,592
Committee of Supply – Head O (Ministry of Health),184,344,528
Committee of Supply – Head W (Ministry of Transport),91,430,521
Committee of Supply – Head I (Ministry of Social and Family Development),170,313,483
Debate on President's Address,101,360,461
